Setup Vector Database (Qdrant)

In [3]:
%pip install -q qdrant-client openai tqdm

In [8]:
import json
import os
from pathlib import Path
from time import sleep
from uuid import NAMESPACE_URL, uuid4, uuid5

import pandas as pd
from openai import OpenAI
from qdrant_client import QdrantClient, models
from tqdm.auto import tqdm

In [26]:
import os
from pathlib import Path
from google.colab import userdata # Import userdata for Colab secrets

# Qdrant
QDRANT_URL = 'https://378d4c2e-957e-43e6-ab03-f4959f4c6dfa.australia-southeast1-0.gcp.cloud.qdrant.io'
# Mengambil QDRANT_API_KEY dari Colab Secrets
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY') # Pastikan Anda telah menyimpan kunci di Colab Secrets dengan nama ini
print(f'QDRANT_API_KEY dimuat: {bool(QDRANT_API_KEY)} (type: {type(QDRANT_API_KEY).__name__})')
COLLECTION_NAME = 'olist_products_rag'

# OpenAI Embedding
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') # Mengambil OPENAI_API_KEY dari Colab Secrets
print(f'OPENAI_API_KEY dimuat: {bool(OPENAI_API_KEY)} (type: {type(OPENAI_API_KEY).__name__})')
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL', 'text-embedding-3-small')

# Setup Google Drive (Colab-friendly)
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
except Exception:
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    print('google.colab tidak terdeteksi. Pastikan path DRIVE_ROOT sesuai environment Anda.')

# Sumber utama: fact table CSV dari Google Drive
FACT_CSV_PATH = Path(
    os.getenv(
        'FACT_CSV_PATH',
        str(
            DRIVE_ROOT
            / 'Purwadhika'
            / 'Final Project'
            / 'Sprint 1 - Data Foundation'
            / '4. Init Projects'
            / 'fact_olist_order_line.csv'
        ),
    )
)

# Sumber opsional/fallback: JSONL siap pakai (boleh dikosongkan)
rag_jsonl_env = os.getenv('RAG_JSONL_PATH', '').strip()
RAG_JSONL_PATH = Path(rag_jsonl_env) if rag_jsonl_env else None

BATCH_SIZE = int(os.getenv('QDRANT_UPSERT_BATCH_SIZE', '64'))

print('COLLECTION_NAME:', COLLECTION_NAME)
print('FACT_CSV_PATH:', FACT_CSV_PATH)
print('RAG_JSONL_PATH (opsional):', RAG_JSONL_PATH)

# Perhatian: Pastikan Anda telah menyimpan QDRANT_API_KEY dan OPENAI_API_KEY yang benar di Colab Secrets.

QDRANT_API_KEY dimuat: True (type: str)
OPENAI_API_KEY dimuat: True (type: str)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
COLLECTION_NAME: olist_products_rag
FACT_CSV_PATH: /content/drive/MyDrive/Purwadhika/Final Project/Sprint 1 - Data Foundation/4. Init Projects/fact_olist_order_line.csv
RAG_JSONL_PATH (opsional): None


In [18]:
def _price_band(value):
    if pd.isna(value):
        return 'unknown'
    if value < 50:
        return 'low'
    if value < 150:
        return 'medium'
    return 'high'

docs = []
source_type = None

# Prioritas 1: build docs langsung dari fact table
if FACT_CSV_PATH.exists():
    df_fact = pd.read_csv(FACT_CSV_PATH)
    required = ['product_id', 'seller_id']
    missing = [c for c in required if c not in df_fact.columns]
    if missing:
        raise ValueError(f'FACT CSV tidak memiliki kolom wajib: {missing}')

    agg = (
        df_fact.groupby(['product_id', 'seller_id'], dropna=False)
        .agg(
            product_category_name_english=('product_category_name_english', 'first'),
            product_category_name=('product_category_name', 'first'),
            seller_city=('seller_city', 'first'),
            llm_product_name_id=('llm_product_name_id', 'first'),
            llm_product_name_en=('llm_product_name_en', 'first'),
            min_price=('item_price', 'min'),
            max_price=('item_price', 'max'),
            avg_price=('item_price', 'mean'),
            avg_review_score=('review_score', 'mean'),
            sample_review=('review_comment_message', 'first'),
        )
        .reset_index()
    )

    for row in agg.to_dict(orient='records'):
        category = row.get('product_category_name_english') or row.get('product_category_name') or 'unknown'
        product_name = row.get('llm_product_name_en') or row.get('llm_product_name_id') or f"Product {row.get('product_id')}"
        avg_price = row.get('avg_price')
        avg_review = row.get('avg_review_score')

        text = (
            f"Product Name: {product_name}\n"
            f"Category: {category}\n"
            f"Seller City: {row.get('seller_city')}\n"
            f"Price Stats: min={row.get('min_price')}, max={row.get('max_price')}, avg={avg_price}\n"
            f"Avg Review Score: {avg_review}\n"
            f"Sample Review: {row.get('sample_review')}"
        )

        docs.append({
            'id': f"{row.get('product_id')}_{row.get('seller_id')}",
            'text': text,
            'metadata': {
                'product_id': row.get('product_id'),
                'seller_id': row.get('seller_id'),
                'category': category,
                'seller_city': row.get('seller_city'),
                'avg_review_score': avg_review,
                'price_band': _price_band(avg_price),
            },
        })

    source_type = 'fact_csv'

# Prioritas 2 (fallback): JSONL jika FACT CSV tidak ada
elif RAG_JSONL_PATH is not None and RAG_JSONL_PATH.exists():
    with RAG_JSONL_PATH.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            docs.append(json.loads(line))
    source_type = 'jsonl'

else:
    raise FileNotFoundError(
        'FACT CSV tidak ditemukan, dan RAG_JSONL_PATH kosong/tidak valid. Setidaknya satu sumber harus tersedia.'
    )

if not docs:
    raise ValueError('Tidak ada dokumen untuk di-index.')

print(f'Sumber dokumen: {source_type}')
print(f'Total dokumen: {len(docs)}')
print('Contoh keys dokumen pertama:', list(docs[0].keys()))

Sumber dokumen: fact_csv
Total dokumen: 34448
Contoh keys dokumen pertama: ['id', 'text', 'metadata']


In [23]:
openai_client = OpenAI(api_key=OPENAI_API_KEY)
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

def extract_text(doc):
    if 'text' in doc and isinstance(doc['text'], str) and doc['text'].strip():
        return doc['text'].strip()
    if 'document' in doc and isinstance(doc['document'], str) and doc['document'].strip():
        return doc['document'].strip()
    raise ValueError('Dokumen tidak memiliki field text/document yang valid.')

def extract_metadata(doc):
    if 'metadata' in doc and isinstance(doc['metadata'], dict):
        return doc['metadata']
    # fallback jika metadata berada di root
    keys = ['product_id', 'seller_id', 'category', 'seller_city', 'avg_review_score', 'price_band']
    return {k: doc.get(k) for k in keys if k in doc}

sample_text = extract_text(docs[0])
sample_embed = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=[sample_text]).data[0].embedding
vector_size = len(sample_embed)
print('Vector size:', vector_size)

Vector size: 1536


## Point 3.1 - Buat Collection Qdrant

In [24]:
collections = qdrant_client.get_collections().collections
collection_names = [c.name for c in collections]

if COLLECTION_NAME not in collection_names:
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE),
    )
    print(f'Collection dibuat: {COLLECTION_NAME}')
else:
    print(f'Collection sudah ada: {COLLECTION_NAME}')

Collection sudah ada: olist_products_rag


## Point 3.2 - Index Dokumen RAG ke Qdrant

In [ ]:
EMBEDDING_BATCH_SIZE = int(os.getenv('EMBEDDING_BATCH_SIZE', '64'))
UPSERT_BATCH_SIZE = int(os.getenv('QDRANT_UPSERT_BATCH_SIZE', '64'))
MAX_RETRIES = int(os.getenv('QDRANT_MAX_RETRIES', '6'))
RESUME_SKIP_EXISTING = os.getenv('QDRANT_RESUME_SKIP_EXISTING', '1') == '1'

def make_point_id(doc):
    raw_id = str(doc.get('id') or uuid4())
    return str(uuid5(NAMESPACE_URL, raw_id))

def with_retry(fn, action_label):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fn()
        except Exception as e:
            if attempt >= MAX_RETRIES:
                raise
            wait_s = min(2 ** attempt, 20)
            print(f'[retry] {action_label} gagal (attempt {attempt}/{MAX_RETRIES}): {e}')
            print(f'[retry] tunggu {wait_s}s lalu coba lagi...')
            sleep(wait_s)

existing_ids = set()
if RESUME_SKIP_EXISTING:
    next_offset = None
    while True:
        points, next_offset = with_retry(
            lambda: qdrant_client.scroll(
                collection_name=COLLECTION_NAME,
                limit=1000,
                with_payload=False,
                with_vectors=False,
                offset=next_offset,
            ),
            'scroll existing points',
        )
        for p in points:
            existing_ids.add(str(p.id))
        if next_offset is None:
            break
    print(f'Existing points di collection: {len(existing_ids)}')

if existing_ids:
    docs_to_process = [d for d in docs if make_point_id(d) not in existing_ids]
else:
    docs_to_process = docs

print(f'Dokumen yang akan diproses: {len(docs_to_process)} / {len(docs)}')

uploaded = 0
for i in tqdm(range(0, len(docs_to_process), EMBEDDING_BATCH_SIZE), desc='Embedding + Upsert'):
    batch_docs = docs_to_process[i:i + EMBEDDING_BATCH_SIZE]
    texts = [extract_text(d) for d in batch_docs]

    emb_resp = with_retry(
        lambda: openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts),
        'create embeddings',
    )
    vectors = [item.embedding for item in emb_resp.data]

    points = []
    for doc, vec in zip(batch_docs, vectors):
        metadata = extract_metadata(doc)
        point_id = make_point_id(doc)
        payload = {
            'text': extract_text(doc),
            'metadata': metadata,
        }
        points.append(models.PointStruct(id=point_id, vector=vec, payload=payload))

    for j in range(0, len(points), UPSERT_BATCH_SIZE):
        chunk = points[j:j + UPSERT_BATCH_SIZE]
        with_retry(
            lambda chunk=chunk: qdrant_client.upsert(collection_name=COLLECTION_NAME, points=chunk),
            'qdrant upsert',
        )
        uploaded += len(chunk)

print(f'Total dokumen ter-upsert di run ini: {uploaded}')
print('Selesai. Jika run terputus, jalankan lagi cell ini untuk melanjutkan tanpa duplikasi.')

Existing points di collection: 25792
Dokumen yang akan diproses: 8656 / 34448


Embedding + Upsert:   0%|          | 0/136 [00:00<?, ?it/s]

Total dokumen ter-upsert di run ini: 8656
Selesai. Jika run terputus, jalankan lagi cell ini untuk melanjutkan tanpa duplikasi.


## Uji Retrieval Sederhana

In [25]:
test_query = 'produk kategori apa dengan review bagus dan harga terjangkau?'
qvec = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=[test_query]).data[0].embedding

if hasattr(qdrant_client, 'query_points'):
    result = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=qvec,
        limit=3,
    )
    hits = result.points
else:
    hits = qdrant_client.search(
        collection_name=COLLECTION_NAME,
        query_vector=qvec,
        limit=3,
    )

for i, h in enumerate(hits, start=1):
    payload = h.payload or {}
    text_preview = str(payload.get('text', ''))[:200]
    print(f'Hit {i} | score={h.score:.4f}')
    print('metadata:', payload.get('metadata', {}))
    print('text preview:', text_preview)
    print('-' * 80)

Hit 1 | score=0.4956
metadata: {'product_id': '9da2cf291ccb36ad01ee959e9ccdbc03', 'seller_id': 'cac4c8e7b1ca6252d8f20b2fc1a2e4af', 'category': 'market_place', 'seller_city': 'indaiatuba', 'avg_review_score': 5.0, 'price_band': 'medium'}
text preview: Product Name: Affordable Marketplace Products
Category: market_place
Seller City: indaiatuba
Price Stats: min=149.99, max=149.99, avg=149.99
Avg Review Score: 5.0
Sample Review: None
--------------------------------------------------------------------------------
Hit 2 | score=0.4732
metadata: {'product_id': '7039b86e688d15677bdccc14562801dc', 'seller_id': '9e9b539eb2806acee3f5c28085c1db9f', 'category': 'small_appliances', 'seller_city': 'contagem', 'avg_review_score': 5.0, 'price_band': 'high'}
text preview: Product Name: Portable Electronic Device
Category: small_appliances
Seller City: contagem
Price Stats: min=159.0, max=159.0, avg=159.0
Avg Review Score: 5.0
Sample Review: None
---------------------------------------------------------